# conv-windowing-1d — ex1: build the 1-D conv window view via as_strided

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-windowing-1d`. Running the final beacon cell reports progress against the `CNN: 1-D conv windowing` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-1d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-1d"
DD_SUBTOPIC = "CNN: 1-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 1-D conv windowing via `as_strided` — quick refresher

A 1-D convolution can be expressed as **two separate steps**:

1. **Window** the input into a strided view of shape `(B, IC, OW, KW)` — each `(KW,)` slice along the new `OW` axis is one kernel-sized window of the original input. The windows overlap.
2. **Einsum** the window against the kernel: `einops.einsum(x_strided, weight, 'b ic ow kw, oc ic kw -> b oc ow')`.

The windowing step is the load-bearing trick. For input strides `(s_b, s_ic, s_w)` and a window count `OW = W - KW + 1` (stride-1 case):

```
x_strided = x.as_strided(
    size=(B, IC, OW, KW),
    stride=(s_b, s_ic, s_w, s_w),  # last two strides are EQUAL: s_w
)
```

The trailing `s_w` on the new `OW` axis means "advance by one element of the original W axis when you move to the next window" — i.e., adjacent windows overlap by `KW - 1` elements. The trailing `s_w` on `KW` walks *within* a window. **No data is copied** — `x_strided` is a view into the same storage as `x`.

**Equivalence check.** The result must agree with `F.conv1d(x, weight)` to floating-point tolerance.

### Exercise 1 — build the 1-D conv window view via as_strided

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `as_strided` to build the `(B, IC, OW, KW)` window view of a 1-D input for stride-1 convolution, then verify that contracting it against a kernel via einsum equals `F.conv1d`.
> Keywords: as_strided, windowing, view, conv1d-equivalence
> ```

**KCs targeted:** `windowing-stride-pattern`, `windowing-output-width`

Implement `ex1_conv1d_windows(x, KW)`. Given input `x` of shape `(B, IC, W)` and kernel width `KW`, return the strided window view of shape `(B, IC, OW, KW)` where `OW = W - KW + 1` and each `(KW,)` slice along the new `OW` axis is one stride-1 window of `x`.

**The trick.** Read `x.stride()` to get `(s_b, s_ic, s_w)`, then call:
```
x.as_strided(
    size=(B, IC, OW, KW),
    stride=(s_b, s_ic, s_w, s_w),
)
```
The trailing pair `(s_w, s_w)` is the key — same stride on the `OW` axis as on the `KW` axis means adjacent windows are offset by 1 element of the original `W` axis (so they overlap by `KW - 1`).

**Constraints.** No copy — your returned tensor must share storage with `x` (the test confirms this).

The verification cell contracts your window view against a random kernel via `einops.einsum(..., 'b ic ow kw, oc ic kw -> b oc ow')` and compares against `F.conv1d`.

In [ ]:
def ex1_conv1d_windows(x: Tensor, KW: int) -> Tensor:
    B, IC, W = x.shape
    OW = W - KW + 1
    s_b, s_ic, s_w = x.stride()
    return x.as_strided(
        size=(B, IC, OW, KW),
        stride=(s_b, s_ic, s_w, s_w),
    )


<details><summary>Solution</summary>

```python
def ex1_conv1d_windows(x: Tensor, KW: int) -> Tensor:
    B, IC, W = x.shape
    OW = W - KW + 1
    s_b, s_ic, s_w = x.stride()
    return x.as_strided(
        size=(B, IC, OW, KW),
        stride=(s_b, s_ic, s_w, s_w),
    )
```

**Reading the stride tuple.** A tensor's `.stride()` returns the number of *elements* (not bytes) you advance through storage when you increment each axis by 1. For a contiguous `(B, IC, W)`, that is `(IC*W, W, 1)` — but you don't need to know this; just read `x.stride()` and reuse the values.

**Why `(s_w, s_w)` on the trailing pair.** The new `OW` axis means 'window index' — stepping by 1 in `OW` must move the window by 1 element of the original `W`, so its stride is `s_w`. The `KW` axis means 'position within a window' — stepping by 1 in `KW` also moves 1 element of `W`, so its stride is also `s_w`. Same stride; different semantics.

**Common pitfall.** If `x` was itself created by striding (e.g. a previous as_strided view), `s_w` may not be `1`. The ARENA code explicitly warns about this — never hardcode `stride=(s_b, s_ic, 1, 1)`. Always read from `x.stride()`.

**No data is copied.** `as_strided` constructs a view header pointing into the same storage. That's why this is fast: the expensive operation is the einsum that follows, not the windowing.

**Generalizing.** Add a `stride` parameter by multiplying the `OW` stride: `stride=(s_b, s_ic, s_w * conv_stride, s_w)`. Add padding by first calling the `conv-padding-zero` drill on `x`, then windowing the padded version. That's the full ARENA 1-D conv in three composable pieces.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()